In [1]:
import pickle

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
from dotenv import load_dotenv

In [4]:
load_dotenv()

True

In [5]:
import numpy as np
import pandas as pd

# Data Preparation

In [6]:
df = pd.read_excel("data.xlsx", header=0)

In [7]:
df.head()

,title_rus,title_eng,goals,tasks,annotation,description,expectations,product_result,result_criterias,social_effect,commercial_effect
0,Исследование приоритетов и механизмов реализац...,Study of Priorities and Mechanisms for Impleme...,Целью проекта является эмпирическая проверка г...,1) Анализ повестки международных доноров\n2) А...,Работа международных фондов (доноров) должна п...,"Согласно определению международных фондов, про...",Аналитический отчет по избранным странам.\n\nС...,NaN,NaN,NaN,NaN
1,Антрополе - научно-популярный видео-подкаст о ...,Anthropole is a Popular Science Video Podcast ...,Выпустить и популяризовать 27 видео-подкастов ...,Снять и смонтировать подкасты;\nРазработать ст...,"\tАнтрополе - научно-популярный проект, в рамк...","Социальное знание близко и интересно обществу,...","Студенты получат опыт монтажа, продвижения и к...",Регулярный видео-подкаст (+экспедиционные филь...,Опубликованы 27 видео-подкастов о необычных со...,Популяризация социального знания должна привес...,Планируется в течение года выйти на окупаемост...
2,"Разработка, создание и ведение сайта, посвящен...","Design, Development and Implementation of a We...","Для того, чтобы получить более полное представ...",Подготовка технического задания для разработчи...,Художественное образование и творчество художн...,Тема обучения арабских художников в художестве...,"Навыки создания сайта (полный цикл, от подгото...","Создание сайта, посвященного истории арабских ...",Выполнение заданий руководителя проекта,Укрепление международных связей с художниками ...,NaN
3,Перевод с английского языка коллективной моног...,Translation from English of the collective mon...,Результатом проекта станет качественный перево...,Результатом проекта станет качественный перево...,"Коллективная монография, авторы которой являют...","Коллективная монография, авторы которой являют...",Участники проекта приобретут новые знания в об...,Результатом проекта станет качественный перево...,Критериями достижения результата будет возможн...,NaN,NaN
4,Сеть военно-политических союзов в Евразии: баз...,Network of Military in Eurasia: a Database,Целью проекта является создание базы данных со...,1.\tОпределение методики включения союзов в ба...,Проект посвящен изучению сети военно-политичес...,Проект посвящен анализу истории существования ...,1.\tОбучение навыкам сбора и анализа данных 2....,База данных союзов 1815-2024 в Евразии.,Создание базы данных как минимум тысячи диадны...,NaN,NaN


In [8]:
df.tail()

,title_rus,title_eng,goals,tasks,annotation,description,expectations,product_result,result_criterias,social_effect,commercial_effect
1253,Влияние мер контроля за движением капитала на ...,The Impact of Capital Controls on the Investme...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1254,"Создание страницы (лендинга) проектов ПУЛ ""Раз...",Creation of a Landing Page for Projects of the...,Основная цель: создать эффективный digital-инс...,Разработать информационную архитектуру и прото...,Проект направлен на разработку и запуск одност...,Проектно-учебная лаборатория «Развитие универс...,Навыки анализа потребностей заказчика.\nОпыт р...,"Рабочий, опубликованный в интернете лендинг.",Функциональные: Все разделы лендинга работают ...,NaN,"Повышение конверсии запросов в контракты, форм..."
1255,Разработка стратегической и контентно-коммуник...,Development of a Strategic and Content-based C...,1. Разработать целостную стратегию присутствия...,Для маркетолога (стратега):\n- провести аудит ...,Проект предполагает разработку системной комму...,Экосистема «Вулканариум–Ойкумена» объединяет н...,Участники:\n- овладеют навыком комплексного ау...,Стратегический документ (дорожная карта на 3–6...,- Наличие полноценного стратегического докумен...,- поддержка просветительской миссии музея «Вул...,- повышение эффективности продвижения услуг (э...
1256,Актив Центра развития карьеры - профориентацио...,Active Club of the Career Development Center -...,Сформировать у студентов Питерской Вышки базов...,1. Разработать и реализовать мероприятия Центр...,Актив Центра развития карьеры — это проектная ...,Актуальность проекта «Актив Центра развития ка...,1. Навыки самостоятельной работы и принятия от...,NaN,NaN,NaN,NaN
1257,Обзор Методологии в Области Регионоведения. Эт...,Area Studies Methodology Overview. Data Collec...,Создать базу данных,Апдейтнуть,Данный этап заключается в поиске и сортировке ...,На текущий момент существует разрозненные пред...,Вклад в формирование базы данных о методология...,NaN,NaN,NaN,NaN


In [9]:
df = df.fillna("")

In [10]:
df.drop_duplicates(inplace=True)

In [11]:
def clean_text(text):

    text = text.replace("\n", " ")
    text = text.replace("\r", " ")
    text = text.replace("\t", " ")

    text = text.lower()

    return text

In [12]:
def clean_title_rus(row):
    return clean_text(row["title_rus"])

In [13]:
df["payload"] = df.apply(clean_title_rus, axis=1)

# Data Modelling

## BM25

In [14]:
import bm25s

resource module not available on Windows


In [15]:
corpus = df["payload"].tolist()

In [16]:
corpus_tokens = bm25s.tokenize(corpus)

In [17]:
retriever = bm25s.BM25(
    corpus=corpus,
    method="robertson",
    k1=1.5,
    b=0.75,
)

In [18]:
retriever.index(corpus_tokens)

In [19]:
def bm25s_search(query: str, top_k: int = 5):
    query_tokens = bm25s.tokenize(query)
    docs_idx, scores = retriever.retrieve(query_tokens, k=top_k)

    docs_idx = docs_idx[0]
    scores = scores[0]

    return pd.DataFrame(data=zip(docs_idx, scores, ["bm25s"] * top_k), columns=["title", "score", "algo"])

In [20]:
for row in bm25s_search("движение капитала", top_k=5).iterrows():
    if row[1]["score"] > 0:
        print(f"{row[1]["algo"]} - {round(row[1]["score"], 2)} - {row[1]["title"]}")
        print()

bm25s - 2.35 - эзотерические практики как форма социального и экономического капитала в россии

bm25s - 2.26 - инвестирование венчурного капитала в ai-стартапы стран брикс: сравнительный анализ стратегических направлений венчурного капитала и влияния практик ai-скрининга на параметры сделок

bm25s - 1.4 - влияние мер контроля за движением капитала на инвестиционный климат и долгосрочный экономический рост в странах с формирующимся рынком (на примере россии, китая и бразилии)



In [21]:
with open("output/bm25s_retriever.pkl", "wb") as f:
    pickle.dump(retriever, f)

##  TF‑IDF + Cosine

In [22]:
from stop_words import get_stop_words

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [24]:
ru_stopwords = get_stop_words("russian")
eng_stopwords = get_stop_words("english")

In [25]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 3),
    min_df=1,
    stop_words=list(set(ru_stopwords + eng_stopwords)),
)

In [26]:
tfidf = vectorizer.fit_transform(df["payload"])

In [27]:
def search_by_query(query: str, top_k: int = 5):
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, tfidf).ravel()
    top_idx = np.argsort(-sims)[:top_k]
    return pd.DataFrame(data=zip(df.iloc[top_idx]["payload"].to_list(), sims[top_idx], ["tfidf"] * top_k), columns=["title", "score", "algo"])


In [28]:
for row in search_by_query("движение капитала", top_k=5).iterrows():
    if row[1]["score"] > 0:
        print(f"{row[1]["algo"]} - {round(row[1]["score"], 2)} - {row[1]["title"]}")
        print()

tfidf - 0.26 - инвестирование венчурного капитала в ai-стартапы стран брикс: сравнительный анализ стратегических направлений венчурного капитала и влияния практик ai-скрининга на параметры сделок

tfidf - 0.23 - эзотерические практики как форма социального и экономического капитала в россии

tfidf - 0.14 - влияние мер контроля за движением капитала на инвестиционный климат и долгосрочный экономический рост в странах с формирующимся рынком (на примере россии, китая и бразилии)



In [29]:
with open("output/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

In [30]:
with open("output/tfidf.pkl", "wb") as f:
    pickle.dump(tfidf, f)

## SentenceTransformers

In [31]:
from sentence_transformers import SentenceTransformer

In [32]:
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10050.05it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [33]:
corpus_texts = df["payload"].tolist()

In [34]:
item_embeddings = model.encode(corpus_texts, normalize_embeddings=True)

In [35]:
def semantic_search(query: str, top_k: int = 5):
    q_emb = model.encode([query], normalize_embeddings=True)
    sims = cosine_similarity(q_emb, item_embeddings).ravel()
    top_idx = np.argsort(-sims)[:top_k]
    return pd.DataFrame(data=zip(df.iloc[top_idx]["payload"].to_list(), sims[top_idx], ["transformer"] * top_k), columns=["title", "score", "algo"])

In [36]:
for row in semantic_search("движение капитала", top_k=3).iterrows():
    if row[1]["score"] > 0:
        print(f"{row[1]["algo"]} - {round(row[1]["score"], 2)} - {row[1]["title"]}")
        print()

transformer - 0.64 - инвестирование венчурного капитала в ai-стартапы стран брикс: сравнительный анализ стратегических направлений венчурного капитала и влияния практик ai-скрининга на параметры сделок

transformer - 0.62 - влияние мер контроля за движением капитала на инвестиционный климат и долгосрочный экономический рост в странах с формирующимся рынком (на примере россии, китая и бразилии)

transformer - 0.54 - политический профайлинг



In [37]:
with open("output/item_embeddings.pkl", "wb") as f:
    pickle.dump(item_embeddings, f)

In [38]:
with open("output/model.pkl", "wb") as f:
    pickle.dump(model, f)

## Ranker

In [39]:
query = "движение капитала"

In [40]:
dfs = []

In [41]:
for foo in [bm25s_search, search_by_query, semantic_search]:
    dfs.append(foo(query, 3))

In [42]:
output = pd.concat(dfs).reset_index(drop=True)

In [43]:
output = output.pivot_table(
    index="title",
    columns="algo",
    values="score",
    fill_value=0,
)

In [44]:
output.columns = ["bm25s", "tfidf", "transformer"]

In [45]:
output.reset_index(inplace=True)

In [46]:
output

,title,bm25s,tfidf,transformer
0,влияние мер контроля за движением капитала на ...,1.400048,0.135269,0.615365
1,инвестирование венчурного капитала в ai-старта...,2.261315,0.257393,0.641289
2,политический профайлинг,0.000000,0.000000,0.538835
3,эзотерические практики как форма социального и...,2.346851,0.226534,0.000000


In [47]:
def simple_rank(candidates_df, w_bm25=0.2, w_tfidf=0.2, w_transformer=0.6):
    df = candidates_df.copy()
    
    for col in ["bm25s", "tfidf", "transformer"]:
        m, M = df[col].min(), df[col].max()
        if M > m:
            df[col + "_norm"] = (df[col] - m) / (M - m)
        else:
            df[col + "_norm"] = 0.0

    df["final_score"] = (
        w_bm25 * df["bm25s_norm"] +
        w_tfidf * df["tfidf_norm"] +
        w_transformer * df["transformer_norm"]
    )

    df = df.sort_values("final_score", ascending=False).reset_index(drop=True)

    return df[["title", "bm25s_norm", "tfidf_norm", "transformer_norm", "final_score"]]

In [48]:
simple_rank(output)

,title,bm25s_norm,tfidf_norm,transformer_norm,final_score
0,инвестирование венчурного капитала в ai-старта...,0.963553,1.000000,1.000000,0.992711
1,влияние мер контроля за движением капитала на ...,0.596565,0.525535,0.959576,0.800166
2,политический профайлинг,0.000000,0.000000,0.840237,0.504142
3,эзотерические практики как форма социального и...,1.000000,0.880111,0.000000,0.376022
